<a href="https://colab.research.google.com/github/jethrocsau/mimic-multimodal-discordance/blob/main/ecg_hubert_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECG modality EDA — HuBERT embeddings
Cohort: first CCU stay with HF ICD, age ≥18, ICU LOS ≥24h, alive at 24h (same as Phase 0).
ECG: nearest 12-lead ECG within **[ICU intime − 72h, +24h)**. Outcome: **1-year mortality**.

Run cells top to bottom. Only two things to edit: `PROJECT` in step 0 and `ID_COL` in step 2.

## 0. Setup

In [3]:
!pip -q install umap-learn

PROJECT  = "YOUR_GCP_PROJECT"    # <- the GCP project you use for BigQuery
EMB_PATH = "/content/drive/MyDrive/ECG_VE_Takeshi/ve_hubert_768.csv"   # shortcut path in your Drive
OUT_DIR  = "/content/drive/MyDrive/capstone_ecg"                       # results saved here

from google.colab import drive, auth
drive.mount("/content/drive")
auth.authenticate_user()

import os, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
os.makedirs(OUT_DIR, exist_ok=True)

ValueError: mount failed

## 1. Cohort + nearest ECG from BigQuery
Self-contained query (no temp tables). The first print should say ~4,496 patients, matching Table A.

In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

SQL = """
WITH hf AS (
  SELECT DISTINCT hadm_id
  FROM `physionet-data.mimiciv_3_1_hosp.diagnoses_icd`
  WHERE (icd_version = 9 AND icd_code LIKE '428%') OR (icd_version = 10 AND icd_code LIKE 'I50%')
),
cohort AS (
  SELECT icu.stay_id, icu.subject_id, icu.hadm_id, icu.intime,
         det.admission_age AS age, det.gender, det.race,
         ROW_NUMBER() OVER (PARTITION BY icu.subject_id ORDER BY icu.intime, icu.stay_id) AS rn
  FROM `physionet-data.mimiciv_3_1_icu.icustays` icu
  JOIN hf ON hf.hadm_id = icu.hadm_id
  JOIN `physionet-data.mimiciv_3_1_derived.icustay_detail` det ON det.stay_id = icu.stay_id
  LEFT JOIN `physionet-data.mimiciv_3_1_hosp.admissions` adm ON adm.hadm_id = icu.hadm_id
  WHERE icu.first_careunit = 'Coronary Care Unit (CCU)'
    AND icu.los >= 1.0
    AND det.admission_age >= 18
    AND NOT (adm.deathtime IS NOT NULL AND adm.deathtime >= icu.intime
             AND adm.deathtime < DATETIME_ADD(icu.intime, INTERVAL 24 HOUR))
),
ecg AS (
  SELECT c.stay_id, e.study_id, e.ecg_time,
         DATETIME_DIFF(e.ecg_time, c.intime, MINUTE) / 60.0 AS offset_h,
         ROW_NUMBER() OVER (PARTITION BY c.stay_id
                            ORDER BY ABS(DATETIME_DIFF(e.ecg_time, c.intime, MINUTE)), e.ecg_time) AS rk
  FROM cohort c
  JOIN `physionet-data.mimiciv_ecg.record_list` e ON e.subject_id = c.subject_id
  WHERE c.rn = 1
    AND e.ecg_time >= DATETIME_SUB(c.intime, INTERVAL 72 HOUR)
    AND e.ecg_time <  DATETIME_ADD(c.intime, INTERVAL 24 HOUR)
)
SELECT c.stay_id, c.subject_id, c.hadm_id, c.intime, c.age, c.gender, c.race,
       (pat.dod IS NOT NULL AND pat.dod <= DATE_ADD(DATE(c.intime), INTERVAL 365 DAY)) AS died_1y,
       ecg.study_id AS ecg_study_id, ecg.ecg_time, ecg.offset_h
FROM cohort c
JOIN `physionet-data.mimiciv_3_1_hosp.patients` pat ON pat.subject_id = c.subject_id
LEFT JOIN ecg ON ecg.stay_id = c.stay_id AND ecg.rk = 1
WHERE c.rn = 1
"""
cohort = client.query(SQL).to_dataframe()
cohort["died_1y"] = cohort["died_1y"].astype(int)
print("patients in cohort:", len(cohort))
print("with ECG in [-72h,+24h):", cohort["ecg_study_id"].notna().sum())
print("1-year mortality:", round(cohort["died_1y"].mean(), 3))
cohort.to_parquet(f"{OUT_DIR}/cohort_ecg_records.parquet")

## 2. Look at the embedding file
Read only 5 rows. Find the column that holds the ECG ID (most likely `study_id`) and set `ID_COL` below.

In [ ]:
import pandas as pd
!ls -lh "{EMB_PATH}"
head = pd.read_csv(EMB_PATH, nrows=5)
print("n columns:", head.shape[1])
print("first columns:", head.columns[:8].tolist())
head.iloc[:, :6]

In [ ]:
ID_COL = "study_id"     # <- change to the actual ID column name after looking above

## 3. Keep only cohort ECGs (chunked read, the full file is large)

In [ ]:
import numpy as np
keep = set(cohort["ecg_study_id"].dropna().astype("int64"))

parts = []
for chunk in pd.read_csv(EMB_PATH, chunksize=50_000):
    ids = pd.to_numeric(chunk[ID_COL], errors="coerce")
    parts.append(chunk[ids.isin(keep)])
emb = pd.concat(parts, ignore_index=True)
emb[ID_COL] = pd.to_numeric(emb[ID_COL]).astype("int64")
emb = emb.drop_duplicates(subset=ID_COL)

# embedding columns = numeric columns that are not IDs
id_like = {ID_COL, "subject_id", "stay_id", "hadm_id", "file_name", "path"}
EMB_COLS = [c for c in emb.columns if c not in id_like and pd.api.types.is_numeric_dtype(emb[c])]
print("embedding dim:", len(EMB_COLS))

df = cohort.dropna(subset=["ecg_study_id"]).copy()
df["ecg_study_id"] = df["ecg_study_id"].astype("int64")
df = df.merge(emb[[ID_COL] + EMB_COLS], left_on="ecg_study_id", right_on=ID_COL, how="inner")

print(f"cohort ECGs: {len(keep)} | matched embeddings: {len(df)} ({len(df)/len(keep):.1%})")
df.to_parquet(f"{OUT_DIR}/ecg_hubert_cohort.parquet")

**Check:** match rate should be close to 100%. If it is much lower, the ID column is wrong or the file covers only part of MIMIC-IV-ECG; stop and check before going on.

## 4. Probe: logistic regression, 5-fold CV, 5 seeds
All models use exactly the same patients, so the AUROCs are directly comparable.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import roc_auc_score, brier_score_loss

y = df["died_1y"].values
X_emb  = df[EMB_COLS].values
X_demo = np.column_stack([df["age"].values, (df["gender"] == "M").astype(int).values])

def run(X, y, seed, tune=True):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    clf = (LogisticRegressionCV(Cs=10, cv=3, scoring="roc_auc", max_iter=3000)
           if tune else LogisticRegression(max_iter=3000))
    p = cross_val_predict(make_pipeline(StandardScaler(), clf), X, y, cv=skf, method="predict_proba")[:, 1]
    brier = brier_score_loss(y, p)
    scaled_brier = 1 - brier / brier_score_loss(y, np.full_like(p, y.mean()))
    return roc_auc_score(y, p), scaled_brier

SEEDS = [0, 1, 2, 3, 4]
rows = []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    y_shuf = rng.permutation(y)
    for name, X, yy, tune in [
        ("Demographics (age + sex)",       X_demo,                     y,      False),
        ("ECG HuBERT",                      X_emb,                      y,      True),
        ("ECG HuBERT + demographics",       np.hstack([X_emb, X_demo]), y,      True),
        ("ECG HuBERT, shuffled labels",     X_emb,                      y_shuf, True),
    ]:
        auc, sb = run(X, yy, seed, tune)
        rows.append({"model": name, "seed": seed, "auroc": auc, "scaled_brier": sb})
        print(f"seed {seed} | {name:32s} AUROC {auc:.3f}")

res = pd.DataFrame(rows)
summary = (res.groupby("model", sort=False)
              .agg(auroc_mean=("auroc", "mean"), auroc_min=("auroc", "min"), auroc_max=("auroc", "max"),
                   scaled_brier_mean=("scaled_brier", "mean"))
              .round(3))
summary.insert(0, "n", len(df)); summary.insert(1, "events", int(y.sum()))
summary.to_csv(f"{OUT_DIR}/ecg_probe_summary.csv")
summary

**How to read it**
- *Shuffled labels* should be ≈ 0.50. Clearly higher means leakage somewhere.
- *Demographics* is the floor. ECG only adds information if it beats this row by more than the seed range.
- *ECG + demographics* vs *Demographics*: how much the ECG adds on top of age and sex.

## 5. UMAP plots

In [ ]:
import umap
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

Z = PCA(n_components=50, random_state=0).fit_transform(StandardScaler().fit_transform(X_emb))
U = umap.UMAP(n_neighbors=30, min_dist=0.3, random_state=0).fit_transform(Z)

age_band = pd.cut(df["age"], [17, 60, 70, 80, 200], labels=["<60", "60-69", "70-79", "80+"])
panels = [("1-year mortality", df["died_1y"].map({0: "alive", 1: "died"})),
          ("Sex", df["gender"]),
          ("Age band", age_band.astype(str))]

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, (title, lab) in zip(axes, panels):
    for g in sorted(lab.dropna().unique()):
        m = (lab == g).values
        ax.scatter(U[m, 0], U[m, 1], s=4, alpha=0.5, label=f"{g} (n={m.sum()})")
    ax.set_title(f"ECG HuBERT UMAP — {title}"); ax.set_xticks([]); ax.set_yticks([])
    ax.legend(markerscale=3, fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ecg_hubert_umap.png", dpi=150)
plt.show()

**How to read it:** look for whether colors form separate regions. Clear grouping by sex or age but not by outcome means the embedding mostly encodes demographics. Weak visual separation by outcome is normal; the probe AUROC is the real test.